In [2]:
import numpy as np

In [3]:
delivery_data = np.genfromtxt(
            'deliveries.csv',
            delimiter= ',',
            dtype=None, 
            names = True,
            missing_values = '',
            filling_values = np.nan, 
            encoding='utf-8'
        )

In [4]:
delivery_data

array([( 335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore',  0, 1, 'SC Ganguly', 'P Kumar', 'BB McCullum', 0, 1, 1, 'legbyes', 0, 'NA', 'NA', 'NA'),
       ( 335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore',  0, 2, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 0, 0, '', 0, 'NA', 'NA', 'NA'),
       ( 335982, 1, 'Kolkata Knight Riders', 'Royal Challengers Bangalore',  0, 3, 'BB McCullum', 'P Kumar', 'SC Ganguly', 0, 1, 1, 'wides', 0, 'NA', 'NA', 'NA'),
       ...,
       (1426312, 2, 'Kolkata Knight Riders', 'Sunrisers Hyderabad', 10, 1, 'VR Iyer', 'Shahbaz Ahmed', 'SS Iyer', 1, 0, 1, '', 0, 'NA', 'NA', 'NA'),
       (1426312, 2, 'Kolkata Knight Riders', 'Sunrisers Hyderabad', 10, 2, 'SS Iyer', 'Shahbaz Ahmed', 'VR Iyer', 1, 0, 1, '', 0, 'NA', 'NA', 'NA'),
       (1426312, 2, 'Kolkata Knight Riders', 'Sunrisers Hyderabad', 10, 3, 'VR Iyer', 'Shahbaz Ahmed', 'SS Iyer', 1, 0, 1, '', 0, 'NA', 'NA', 'NA')],
      shape=(260920,), dtype=[('match_id', '<i8'), ('i

In [5]:
delivery_data.dtype.names

('match_id',
 'inning',
 'batting_team',
 'bowling_team',
 'over',
 'ball',
 'batter',
 'bowler',
 'non_striker',
 'batsman_runs',
 'extra_runs',
 'total_runs',
 'extras_type',
 'is_wicket',
 'player_dismissed',
 'dismissal_kind',
 'fielder')

In [6]:
match_id = delivery_data['match_id'][:10]
match_id

array([335982, 335982, 335982, 335982, 335982, 335982, 335982, 335982,
       335982, 335982])

In [7]:
batting_team = delivery_data['batting_team'][:10]
batting_team

array(['Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders',
       'Kolkata Knight Riders', 'Kolkata Knight Riders'], dtype='<U27')

In [8]:
batter = delivery_data['batter'][:10]
batter

array(['SC Ganguly', 'BB McCullum', 'BB McCullum', 'BB McCullum',
       'BB McCullum', 'BB McCullum', 'BB McCullum', 'BB McCullum',
       'BB McCullum', 'BB McCullum'], dtype='<U23')

In [9]:
bowler = delivery_data['bowler'][:10]
bowler

array(['P Kumar', 'P Kumar', 'P Kumar', 'P Kumar', 'P Kumar', 'P Kumar',
       'P Kumar', 'Z Khan', 'Z Khan', 'Z Khan'], dtype='<U23')

In [10]:
batsman_runs = delivery_data['batsman_runs'][:10]
batsman_runs

array([0, 0, 0, 0, 0, 0, 0, 0, 4, 4])

In [11]:
over = delivery_data['over'][:10]
over

array([0, 0, 0, 0, 0, 0, 0, 1, 1, 1])

### Total runs scored in each match

In [12]:
match_id = delivery_data['match_id']
batsman_runs = delivery_data['batsman_runs']

In [13]:
unique_matches = np.unique(match_id)

In [14]:
total_runs = []

for m in unique_matches:
    total_runs.append(batsman_runs[match_id == m].sum())

total_runs = np.array(total_runs)

print(total_runs)

# match_id_shifted = match_id - match_id.min()

#total_runs = np.bincount(match_id_shifted, weights=batsman_runs)

[268 430 244 ... 336 301 203]


In [15]:
result = np.column_stack((unique_matches, total_runs))
print(result)

[[ 335982     268]
 [ 335983     430]
 [ 335984     244]
 ...
 [1426310     336]
 [1426311     301]
 [1426312     203]]


### Top 5 batters based on total runs

In [16]:
batsman = delivery_data['batter']
batsman_runs = delivery_data['batsman_runs']

In [17]:
# Getting unique batters
unique_batters = np.unique(batsman)

In [18]:
# Computing total runs for each batter
total_runs = []

for b in unique_batters:
    runs = batsman_runs[batsman == b].sum()
    total_runs.append(runs)

total_runs = np.array(total_runs)

In [19]:
# indices of top 5 batters
top5_idx = np.argsort(total_runs)[-5:][::-1]

# top 5 batters and their runs
top5_batters = unique_batters[top5_idx]
top5_runs = total_runs[top5_idx]

# Display result
for i in range(5):
    print(top5_batters[i], ":", int(top5_runs[i]))

V Kohli : 8014
S Dhawan : 6769
RG Sharma : 6630
DA Warner : 6567
SK Raina : 5536


In [20]:
# Different Approach

# Sort by batsman
sorted_idx = np.argsort(batsman)
batsman_sorted = batsman[sorted_idx]
runs_sorted = batsman_runs[sorted_idx]

# unique batters and their start indices
unique_batters, indices = np.unique(batsman_sorted, return_index=True)

# Sum runs using split
total_runs = np.add.reduceat(runs_sorted, indices)

# Top 5
top5_idx = np.argsort(total_runs)[-5:][::-1]

for i in top5_idx:
    print(unique_batters[i], ":", int(total_runs[i]))

V Kohli : 8014
S Dhawan : 6769
RG Sharma : 6630
DA Warner : 6567
SK Raina : 5536


### Computing strike rate

In [22]:
batsman = delivery_data['batter']
batsman_runs = delivery_data['batsman_runs']

In [23]:
# Sort by batsman (required for grouping)
sorted_idx = np.argsort(batsman)

batsman_sorted = batsman[sorted_idx]
runs_sorted = batsman_runs[sorted_idx]

In [24]:
# Unique batters + group indices
unique_batters, indices = np.unique(batsman_sorted, return_index=True)

In [25]:
# Total runs per batter
total_runs = np.add.reduceat(runs_sorted, indices)

In [26]:
# Balls faced = count of deliveries
balls_faced = np.diff(np.append(indices, len(batsman_sorted)))

In [29]:
# Strike rate
strike_rate = (total_runs / balls_faced) * 100

In [31]:
for i in range(20):
    print(unique_batters[i], round(strike_rate[i], 2))

A Ashish Reddy 142.86
A Badoni 125.54
A Chandila 57.14
A Chopra 70.67
A Choudhary 125.0
A Dananjaya 80.0
A Flintoff 108.77
A Kamboj 100.0
A Kumble 71.43
A Manohar 127.62
A Mishra 86.59
A Mithun 130.77
A Mukund 82.61
A Nehra 65.08
A Nortje 96.08
A Raghuvanshi 149.54
A Singh 20.0
A Symonds 124.71
A Tomar 50.0
A Uniyal 57.14


### Economy Rate of Bowlers

In [32]:
bowler = delivery_data['bowler']
total_runs = delivery_data['total_runs']

In [33]:
# Sort by bowler
sorted_idx = np.argsort(bowler)

In [34]:
bowler_sorted = bowler[sorted_idx]
runs_sorted = total_runs[sorted_idx]

# Unique bowlers + indices
unique_bowlers, indices = np.unique(bowler_sorted, return_index=True)

In [35]:
# Total runs conceded
runs_conceded = np.add.reduceat(runs_sorted, indices)

# Balls bowled (count rows)
balls_bowled = np.diff(np.append(indices, len(bowler_sorted)))

In [36]:
# Economy rate
economy = (runs_conceded / balls_bowled) * 6

In [37]:
for i in range(20):
    print(unique_bowlers[i], round(economy[i], 2))

A Ashish Reddy 8.89
A Badoni 8.88
A Chandila 6.28
A Choudhary 8.0
A Dananjaya 11.28
A Flintoff 9.64
A Kamboj 10.15
A Kumble 6.65
A Mishra 7.3
A Mithun 9.17
A Nehra 7.71
A Nel 10.33
A Nortje 8.83
A Singh 7.89
A Symonds 7.71
A Uniyal 10.58
A Zampa 7.93
AA Chavan 7.98
AA Jhunjhunwala 8.86
AA Kazi 9.69


### Average runs per over

In [38]:
over = delivery_data['over']
batsman_runs = delivery_data['batsman_runs']

In [39]:
# Sort by over
sorted_idx = np.argsort(over)

over_sorted = over[sorted_idx]
runs_sorted = batsman_runs[sorted_idx]

In [40]:
# Unique overs + indices
unique_overs, indices = np.unique(over_sorted, return_index=True)

In [41]:
# Total runs per over
total_runs = np.add.reduceat(runs_sorted, indices)

# Balls per over
balls = np.diff(np.append(indices, len(over_sorted)))

In [42]:
# Average runs per ball in each over
avg_runs = total_runs / balls

In [44]:
for i in range(len(unique_overs)):
    print("Over", unique_overs[i] + 1, ":", round(avg_runs[i], 2))

Over 1 : 0.89
Over 2 : 1.08
Over 3 : 1.25
Over 4 : 1.29
Over 5 : 1.31
Over 6 : 1.31
Over 7 : 1.04
Over 8 : 1.14
Over 9 : 1.19
Over 10 : 1.17
Over 11 : 1.21
Over 12 : 1.24
Over 13 : 1.24
Over 14 : 1.29
Over 15 : 1.33
Over 16 : 1.37
Over 17 : 1.42
Over 18 : 1.5
Over 19 : 1.56
Over 20 : 1.67
